# 5.2 การจัดสรรแบบมีข้อจำกัด และการจำลอง

แผนทั้งหมดสร้างด้วย `python -m chapter5.build --plans` (ขนานหลายกระบวนการ) และ `--uncertainty`
สมุดบันทึกนี้ (1) แก้ปัญหาแผน P-R ใหม่หนึ่งแผนให้เห็นขั้นตอน และตรวจว่าตรงกับแผนที่เก็บไว้ (2) แสดงสมุดบัญชีรายจังหวัด-ปี (3) แสดงผลภายใต้ความไม่แน่นอน

**ตัวแปรตัดสินใจ:** รับใหม่ $x_{pt}$ และโยกย้าย $z_{pt}$ เป็นจำนวนเต็ม
**สมการคงอยู่:** $H_{pt}=(1-d_p+r_p)H_{p,t-1}+x_{pt}+z^{in}_{pt}-z^{out}_{pt}$
**เป้าหมายหลัก:** $J=\sum_t\sum_p (T_{pt}/\sum_q T_{qt})\,s_{pt}^2,\; s\ge 1-H/T$ แล้วจึงลดความไม่สมดุลของการกระจาย และลดการโยกย้าย ภายในความคลาดเคลื่อน 0.05%

In [1]:
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "chapter5.toml").exists())
sys.path.insert(0, str(_root / "src"))
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image
pd.options.display.max_columns = 40
pd.options.display.width = 200
from chapter5 import config as C, pipeline as PL, allocation as AL, simulation as SIM, responsibilities as R
S = C.load()
D = S.dirs()
tab = lambda n, f="tables": pd.read_csv(D[f] / f"{n}.csv", encoding="utf-8-sig")
print("scenario:", S.name, "| config_hash:", S.hash())

scenario: canonical | config_hash: 9634ed4e506743c4


In [2]:
ctx = PL.context('{}', '{}')
res = {r['job_id']: r for r in PL.load_plan_results(S)}
job = [j for j in PL.catalogue(S, ('principal',)) if j['job_id'] == 'P-R'][0]
new = PL.run_job(job)
print(new['status'], [ (s['stage'], round(s['objective'], 5), s.get('outcome')) for s in new['stages']])
print('identical to the cached plan:', (new['x'] == res['P-R']['x']).all())
print('constraint check:', new['check'])

exploratory_policy_scenario [('shortfall', 4.35068, 'optimal within gap'), ('distribution', 1.08933, 'optimal within gap')]
identical to the cached plan: True
constraint check: {'pool_conservation': True, 'integer': True, 'floor': True, 'nonnegative_stock': True, 'transfer_balance': True, 'cap': True, 'all_ok': True, 'no_simultaneous_in_and_out': True}


## แผนทั้งหมดและสถานะ

In [3]:
display(tab('T5_PLAN_INDEX', 'validation')[['plan_id', 'group', 'status', 'cap_mode', 'all_constraints_ok', 'max_mip_gap', 'transfer_events', 'stock_2040', 'stage_outcomes']])

,plan_id,group,status,cap_mode,all_constraints_ok,max_mip_gap,transfer_events,stock_2040,stage_outcomes
0,P-RT,principal,exploratory_policy_scenario,gross_arrivals,True,0.003962,0.0,30010.616183,shortfall: time limit: best incumbent; recruit...
1,S-RT,principal,exploratory_policy_scenario,gross_arrivals,True,0.007130,0.0,29976.479078,shortfall: time limit: best incumbent; recruit...
2,N-RT,principal,exploratory_policy_scenario,gross_arrivals,True,0.000116,0.0,30051.110384,shortfall: optimal within gap; distribution: n...
3,S-RT|B,principal,exploratory_policy_scenario,gross_arrivals,True,0.006665,0.0,29957.732574,shortfall: time limit: best incumbent; recruit...
4,N-RT|B,principal,exploratory_policy_scenario,gross_arrivals,True,0.000076,0.0,30053.025133,shortfall: optimal within gap; distribution: n...
...,...,...,...,...,...,...,...,...,...
75,S-R|greedy,benchmark,exploratory_policy_scenario,gross_arrivals,True,NaN,0.0,29907.884197,NaN
76,N-R|greedy,benchmark,exploratory_policy_scenario,gross_arrivals,True,NaN,0.0,29986.714609,NaN
77,P-R|tangent_fine,benchmark,exploratory_policy_scenario,gross_arrivals,True,0.000124,0.0,30003.827814,shortfall: optimal within gap; distribution: o...
78,INV|gross_arrivals,benchmark,solved,NaN,NaN,NaN,NaN,NaN,NaN


## สมุดบัญชีรายจังหวัด-ปี (แผน N-R ปี 2569)

In [4]:
L = res['N-R']['ledger']
cols = ['province', 'opening_headcount', 'expected_resignations', 'expected_retirements', 'expected_departures_total', 'expected_reentries', 'new_appointments_accepted_starts', 'expected_closing_stock', 'target_own', 'attainment_own', 'cap_value', 'cap_binding', 'allocation_reason']
display(L[L.year_ce == 2026][cols].sort_values('attainment_own').head(20))

,province,opening_headcount,expected_resignations,expected_retirements,expected_departures_total,expected_reentries,new_appointments_accepted_starts,expected_closing_stock,target_own,attainment_own,cap_value,cap_binding,allocation_reason
25,บึงกาฬ,72.0,8.978419,0.489473,9.619392,0.358593,6,68.739201,424.863392,0.161791,6.0,True,onboarding cap binds (allocation limited by th...
26,หนองบัวลำภู,113.0,9.606741,0.758872,10.595944,0.596078,10,113.000134,501.380268,0.225378,10.0,True,onboarding cap binds (allocation limited by th...
0,สมุทรปราการ,291.0,19.516185,2.294722,22.635568,2.093606,27,297.458038,1286.049822,0.231296,27.0,True,onboarding cap binds (allocation limited by th...
2,ปทุมธานี,279.0,12.951797,2.140294,15.877571,1.525535,26,290.647964,1152.686576,0.252148,26.0,True,onboarding cap binds (allocation limited by th...
16,สระแก้ว,154.0,17.685132,1.053282,19.571476,0.674247,14,149.102771,522.342949,0.285450,14.0,True,onboarding cap binds (allocation limited by th...
24,อำนาจเจริญ,103.0,5.745414,0.706493,6.666976,0.384100,9,105.717124,369.558708,0.286063,9.0,True,onboarding cap binds (allocation limited by th...
33,กาฬสินธุ์,292.0,13.411970,1.920153,16.144442,1.126294,28,304.981852,1047.925097,0.291034,28.0,True,onboarding cap binds (allocation limited by th...
15,นครนายก,76.0,5.715540,0.551135,6.431593,0.243510,7,76.811917,254.712774,0.301563,7.0,True,onboarding cap binds (allocation limited by th...
48,กำแพงเพชร,209.0,8.469597,1.341196,10.278530,0.704020,20,219.425491,716.645966,0.306184,20.0,True,onboarding cap binds (allocation limited by th...
53,เพชรบูรณ์,283.0,18.696231,1.878617,21.267051,0.845384,27,289.578332,942.526828,0.307236,27.0,True,onboarding cap binds (allocation limited by th...


## ดุลระดับประเทศ (ต้องปิดยอดเป็นศูนย์)

In [5]:
nb_ = pd.read_csv(D['plans'] / 'chapter5_national_annual_balance.csv', encoding='utf-8-sig')
display(nb_[nb_.plan_id.isin(['P-R', 'N-RT'])])
print('max reconciliation difference:', nb_.reconciliation_difference.abs().max())

,plan_id,year_ce,opening_stock,new_appointments,transfers_counted_once,transfers_in,expected_departures,expected_resignations,expected_retirements,expected_other_departures,expected_reentries,closing_stock,target_own,population_civil,plan_status,available,unplaced_balance,reconciliation_difference,transfer_balance_difference,national_attainment_own,year_be
15,N-RT,2026,22685.000000,2000,0,0,1418.379582,1192.785695,166.017225,59.576663,107.512034,23374.132452,59197.545324,5.919755e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.394850,2569
16,N-RT,2027,23374.132452,2000,0,0,1453.714310,1221.449890,170.954098,61.310322,110.324494,24030.742636,59061.645768,5.906165e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.406876,2570
17,N-RT,2028,24030.742636,2000,0,0,1488.013161,1249.415225,175.627847,62.970089,113.128962,24655.858437,58906.336845,5.890634e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.418560,2571
18,N-RT,2029,24655.858437,2000,0,0,1518.864894,1274.453320,180.009657,64.401917,115.621672,25252.615215,58731.302436,5.873130e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.429969,2572
19,N-RT,2030,25252.615215,2000,0,0,1551.102381,1300.957252,184.182479,65.962649,118.406299,25819.919134,58535.719510,5.853572e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.441097,2573
20,N-RT,2031,25819.919134,2000,0,0,1582.819033,1327.206871,188.170912,67.441250,121.166567,26358.266668,58319.371353,5.831937e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.451964,2574
21,N-RT,2032,26358.266668,2000,0,0,1611.630880,1350.963536,191.978324,68.689020,123.576311,26870.212098,58081.603211,5.808160e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.462629,2575
22,N-RT,2033,26870.212098,2000,0,0,1641.953777,1376.196353,195.575015,70.182408,126.206408,27354.464730,57822.399783,5.782240e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.473077,2576
23,N-RT,2034,27354.464730,2000,0,0,1670.899696,1400.435859,199.095649,71.368188,128.652862,27812.217896,57542.257740,5.754226e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.483336,2577
24,N-RT,2035,27812.217896,2000,0,0,1699.464372,1424.422458,202.326502,72.715412,130.875460,28243.628984,57240.652896,5.724065e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.493419,2578


max reconciliation difference: 3.637978807091713e-12


## การทดลองป้อนกลับ N-FB

In [6]:
display(tab('T5_FEEDBACK_LOG'))
display(tab('T5_FEEDBACK_CONVERGENCE').query('year_ce in [2026, 2033, 2040]'))

,year,eta_mean,eta_min,eta_max,gain_B_mean,gain_B_max,gain_A_mean,provinces_gain_B_ge_1,plan_id,lam
0,2026,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,NFB0-R,0.0
1,2027,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,NFB0-R,0.0
2,2028,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,NFB0-R,0.0
3,2029,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,NFB0-R,0.0
4,2030,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,NFB0-R,0.0
...,...,...,...,...,...,...,...,...,...,...
70,2036,0.296356,0.244969,0.340184,0.598604,1.418590,0.587659,1,NFB1-R|grow,1.0
71,2037,0.292907,0.238765,0.339199,0.579942,1.423138,0.569343,1,NFB1-R|grow,1.0
72,2038,0.290089,0.233263,0.339280,0.563346,1.424741,0.553048,1,NFB1-R|grow,1.0
73,2039,0.286731,0.227813,0.339218,0.546405,1.422916,0.536418,1,NFB1-R|grow,1.0


,plan_id,year_ce,gini_vs_nstd,bottom_quartile_vs_nstd,share_receipts_top19_2025,own_target_total,own_attainment,own_gini
0,N-R,2026,0.114829,0.300006,0.2050,59197.545324,0.394850,0.114829
7,N-R,2033,0.091668,0.350491,0.1865,57822.399783,0.472589,0.091668
14,N-R,2040,0.073459,0.395006,0.2170,55406.781683,0.541616,0.073459
15,NFB0-R,2026,0.114830,0.300006,0.2050,59197.545324,0.394850,0.114830
22,NFB0-R,2033,0.090701,0.350491,0.1725,57822.399783,0.472344,0.090701
29,NFB0-R,2040,0.076765,0.395006,0.1575,55406.781683,0.540807,0.076765
30,NFB0.5-R,2026,0.114830,0.300006,0.2050,59197.545324,0.394850,0.087422
37,NFB0.5-R,2033,0.098838,0.350491,0.2010,57822.399783,0.472485,0.068591
44,NFB0.5-R,2040,0.087506,0.399529,0.1550,55406.781683,0.540582,0.058176
45,NFB1-R,2026,0.116715,0.300006,0.2270,59197.545324,0.394850,0.069087


## ความไม่แน่นอน: แผนตายตัว การปรับแผนรายปี และขอบเขตรู้อนาคต

In [7]:
u = tab('T5_UNCERTAINTY_SUMMARY')
display(u[u.metric.isin(['bottom_quartile_2040', 'cumulative_j'])])
display(tab('T5_UNCERTAINTY_OPERATIONS'))
display(tab('T5_PAIRED_DIFFERENCES', 'validation'))

,plan_id,mode,criterion,metric,mean,p05,p50,p95,n_valid
1,N-R,fixed_schedule,N,bottom_quartile_2040,0.389538,0.368596,0.389458,0.410420,486
4,N-R,fixed_schedule,N,cumulative_j,4.343863,4.205518,4.347123,4.482297,486
7,N-R,fixed_schedule,P,bottom_quartile_2040,0.379146,0.362525,0.378670,0.395361,500
10,N-R,fixed_schedule,P,cumulative_j,4.354914,4.213694,4.357032,4.489046,500
13,N-R,fixed_schedule,S,bottom_quartile_2040,0.389584,0.371822,0.389780,0.407100,500
...,...,...,...,...,...,...,...,...,...
184,P-R,fixed_schedule_same_paths,P,cumulative_j,4.330301,4.198922,4.342965,4.459702,60
187,S-R,fixed_schedule_same_paths,S,bottom_quartile_2040,0.410911,0.395938,0.411256,0.429654,60
190,S-R,fixed_schedule_same_paths,S,cumulative_j,4.272690,4.142939,4.289099,4.399665,60
193,N-R,fixed_schedule_same_paths,N,bottom_quartile_2040,0.386893,0.367101,0.385799,0.407967,60


,plan_id,cancelled_mean,cap_breach_province_years_mean,share_paths_any_cap_breach,protection_breach_mean
0,N-R,0.0,237.928,1.0,0.0
1,N-RT,0.0,268.404,1.0,0.0
2,NFB1-R,0.0,236.564,1.0,0.0
3,P-R,0.0,239.928,1.0,0.0
4,P-RT,0.0,239.928,1.0,0.0
5,S-R,0.0,189.908,1.0,0.0
6,S-RT,0.0,189.908,1.0,0.0
7,SQ-C,0.0,145.488,1.0,0.0


,comparison,criterion,mean,mean_first_half,batch_se,p05,p95,share_positive,n
0,N-R − P-R,P (bottom-quartile attainment 2040),0.000041,0.000278,0.000151,-0.007277,0.006540,0.474,500
1,S-R − P-R,P (bottom-quartile attainment 2040),-0.029869,-0.029645,0.000238,-0.039339,-0.019286,0.000,500
2,P-RT − P-R,P (bottom-quartile attainment 2040),0.000000,0.000000,0.000000,0.000000,0.000000,0.000,500
3,SQ-C − P-R,P (bottom-quartile attainment 2040),-0.067220,-0.066974,0.000289,-0.079018,-0.053906,0.000,500
